In [1]:
print("""
@File         : performing_one-hot_encoding_of_frequent_categories.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-31 21:04:14
@Email        : cuixuanstephen@gmail.com
@Description  : 对频繁类别执行独热编码
""")


@File         : performing_one-hot_encoding_of_frequent_categories.ipynb
@Author(s)    : Stephen CUI
@LastEditor(s): Stephen CUI
@CreatedTime  : 2025-01-31 21:04:14
@Email        : cuixuanstephen@gmail.com
@Description  : 对频繁类别执行独热编码



独热编码用二进制变量表示每个变量的类别。因此，对具有多个分类特征的高基数变量或数据集进行独热编码可以显著扩展特征空间。这反过来可能会增加使用机器学习模型的计算成本或降低其性能。为了减少二进制变量的数量，我们可以对最频繁的类别进行独热编码。对顶级类别进行独热编码相当于将剩余的、不太频繁的类别视为一个唯一的类别。

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
data = pd.read_csv("credit_approval_uci.csv")
X_train, X_test, y_train, y_test = train_test_split(
    data.drop(labels=["target"], axis=1),
    data["target"],
    test_size=0.3,
    random_state=0,
)

> 需要在训练集中确定最常见的类别。这是为了避免数据泄露。

In [4]:
X_train['A6'].unique()

array(['c', 'q', 'w', 'ff', 'm', 'i', 'e', 'cc', 'x', 'd', 'k', 'j',
       'Missing', 'aa', 'r'], dtype=object)

In [6]:
X_train['A6'].value_counts().sort_values(ascending=False).head(5)

A6
c     93
q     56
w     48
i     41
ff    38
Name: count, dtype: int64

In [9]:
top_5 = [x for x in X_train['A6'].value_counts().sort_values(ascending=False).head(5).index]

# X_train['A6'].value_counts().sort_values(ascending=False).head(5).index.to_list()

['c', 'q', 'w', 'i', 'ff']

In [10]:
X_train_enc = X_train.copy()
X_test_enc = X_test.copy()

for label in top_5:
    X_train_enc[f'A6_{label}'] = np.where(X_train['A6'] == label, 1, 0)
    
    X_test_enc[f'A6_{label}'] = np.where(X_test['A6'] == label, 1, 0)

In [11]:
X_train_enc[['A6'] + [f'A6_{label}' for label in top_5]].head(10)

,A6,A6_c,A6_q,A6_w,A6_i,A6_ff
596,c,1,0,0,0,0
303,q,0,1,0,0,0
204,w,0,0,1,0,0
351,ff,0,0,0,0,1
118,m,0,0,0,0,0
247,q,0,1,0,0,0
652,i,0,0,0,1,0
513,e,0,0,0,0,0
230,cc,0,0,0,0,0
250,e,0,0,0,0,0


In [26]:
from sklearn.preprocessing import OneHotEncoder

In [32]:
encoder = OneHotEncoder(min_frequency=39, max_categories=6, sparse_output=False).set_output(transform="pandas")

In [33]:
X_train_enc = encoder.fit_transform(X_train[['A6', 'A7']])

In [34]:
X_test_enc = encoder.transform(X_test[['A6', 'A7']])

In [35]:
X_train_enc.head()

,A6_c,A6_i,A6_q,A6_w,A6_infrequent_sklearn,A7_bb,A7_ff,A7_h,A7_v,A7_infrequent_sklearn
596,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
303,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
204,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
351,0.0,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
118,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


> The number of frequent categories to encode is arbitrarily determined by the user.

In [37]:
from feature_engine.encoding import OneHotEncoder

ohe_enc = OneHotEncoder(top_categories=5, variables=['A6', 'A7'])

In [38]:
ohe_enc.fit(X_train)

OneHotEncoder(top_categories=5, variables=['A6', 'A7'])

In [39]:
X_train_enc = ohe_enc.transform(X_train)

X_test_enc = ohe_enc.transform(X_test)

In [40]:
ohe_enc.encoder_dict_

{'A6': ['c', 'q', 'w', 'i', 'ff'], 'A7': ['v', 'h', 'ff', 'bb', 'z']}